# 🕐 MindTrack AI v5 — Temporal Mamba SSM Training

**Model:** Mamba-style Selective State Space Model (4-layer SSM + FFN)

**Approach:** Extract temporal feature sequences from RAVDESS + CREMA-D audio.
Each audio clip → sliding windows of mel-spectrogram features → temporal sequence → emotion class.

This trains the model on **real temporal patterns** in speech emotion, not synthetic data.

**Output:** `temporal_mamba.pt` (~1 MB)

In [ ]:
# Step 0: Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 1: Install dependencies
!pip install -q librosa soundfile kaggle scikit-learn matplotlib seaborn tqdm

In [ ]:
# Step 2: Kaggle credentials
import os
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'  # <-- CHANGE THIS
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY'  # <-- CHANGE THIS

In [ ]:
# Step 3: Download datasets
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio -p /content/data --unzip
!kaggle datasets download -d ejlok1/cremad -p /content/data/cremad --unzip
!kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess -p /content/data/tess --unzip
print('\n✅ Datasets downloaded!')

In [ ]:
import os
import glob
import json
import math
import logging
from pathlib import Path

import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

EMOTION_LABELS = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
NUM_CLASSES = len(EMOTION_LABELS)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Step 4: Load file paths (same loaders as voice notebook)

def load_ravdess(data_dir='/content/data'):
    files, labels = [], []
    for wav in glob.glob(os.path.join(data_dir, '**/*.wav'), recursive=True):
        parts = os.path.basename(wav).split('-')
        if len(parts) >= 3:
            try:
                ec = int(parts[2]) - 1
                if 0 <= ec < NUM_CLASSES:
                    files.append(wav); labels.append(ec)
            except ValueError: pass
    print(f'RAVDESS: {len(files)}')
    return files, labels

def load_cremad(data_dir='/content/data/cremad'):
    MAP = {'ANG': 4, 'DIS': 6, 'FEA': 5, 'HAP': 2, 'NEU': 0, 'SAD': 3}
    files, labels = [], []
    for wav in glob.glob(os.path.join(data_dir, '**/*.wav'), recursive=True):
        parts = os.path.basename(wav).replace('.wav','').split('_')
        if len(parts)>=3 and parts[2] in MAP:
            files.append(wav); labels.append(MAP[parts[2]])
    print(f'CREMA-D: {len(files)}')
    return files, labels

def load_tess(data_dir='/content/data/tess'):
    MAP = {'angry':4,'disgust':6,'fear':5,'happy':2,'neutral':0,'ps':7,'sad':3}
    files, labels = [], []
    for wav in glob.glob(os.path.join(data_dir, '**/*.wav'), recursive=True):
        parts = os.path.basename(wav).lower().replace('.wav','').split('_')
        if parts[-1] in MAP:
            files.append(wav); labels.append(MAP[parts[-1]])
    print(f'TESS: {len(files)}')
    return files, labels

r_f, r_l = load_ravdess()
c_f, c_l = load_cremad()
t_f, t_l = load_tess()
all_files = r_f + c_f + t_f
all_labels = r_l + c_l + t_l
print(f'\nTotal: {len(all_files)} files')

In [ ]:
# Step 5: Extract temporal feature sequences from audio
# Each audio → split into N overlapping windows → extract features per window → temporal sequence

SEQ_LEN = 10  # number of temporal windows
FEATURE_DIM = 7  # matches the inference code's input_dim=7

def extract_temporal_features(audio_path, sr=16000, seq_len=SEQ_LEN):
    """Extract a sequence of features from sliding windows over audio.
    
    Features per window (7 dims, matching inference code):
      0: energy (RMS) → proxy for stress
      1: spectral centroid → proxy for anxiety
      2: spectral rolloff → proxy for stability (inverted)
      3: zero crossing rate → proxy for depression_risk
      4: spectral bandwidth → proxy for overall_risk
      5: fundamental frequency → voice_freq
      6: MFCC variance → confidence
    """
    try:
        y, sr = librosa.load(audio_path, sr=sr, mono=True)
    except Exception:
        return None
    
    if len(y) < sr * 0.5:  # skip very short clips
        return None
    
    # Split audio into seq_len overlapping windows
    window_size = len(y) // seq_len
    hop = max(1, (len(y) - window_size) // max(1, seq_len - 1))
    
    features = []
    for i in range(seq_len):
        start = i * hop
        end = min(start + window_size, len(y))
        segment = y[start:end]
        
        if len(segment) < 256:
            segment = np.pad(segment, (0, 256 - len(segment)))
        
        # Extract features
        rms = float(np.sqrt(np.mean(segment**2)))
        
        spec_cent = librosa.feature.spectral_centroid(y=segment, sr=sr)
        centroid = float(np.mean(spec_cent)) / 4000.0  # normalize
        
        spec_roll = librosa.feature.spectral_rolloff(y=segment, sr=sr)
        rolloff = float(np.mean(spec_roll)) / 8000.0
        
        zcr = librosa.feature.zero_crossing_rate(segment)
        zcr_val = float(np.mean(zcr))
        
        spec_bw = librosa.feature.spectral_bandwidth(y=segment, sr=sr)
        bandwidth = float(np.mean(spec_bw)) / 4000.0
        
        # Fundamental frequency
        try:
            pitches, magnitudes = librosa.piptrack(y=segment, sr=sr)
            pitch_vals = pitches[magnitudes > np.median(magnitudes)]
            pitch_vals = pitch_vals[pitch_vals > 50]
            f0 = float(np.median(pitch_vals)) / 500.0 if len(pitch_vals) > 0 else 0.36
        except:
            f0 = 0.36
        
        # MFCC variance as confidence proxy
        mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
        mfcc_var = float(np.var(mfcc)) / 100.0
        
        features.append([rms, centroid, rolloff, zcr_val, bandwidth, f0, min(1.0, mfcc_var)])
    
    return np.array(features, dtype=np.float32)

# Test on one file
test_feat = extract_temporal_features(all_files[0])
if test_feat is not None:
    print(f'Feature shape: {test_feat.shape}')  # (10, 7)
    print(f'Sample features: {test_feat[0]}')

In [ ]:
# Step 6: Extract features for all files (cached)

print('Extracting temporal features from all audio files...')
sequences = []
seq_labels = []
skipped = 0

for i, (fp, label) in enumerate(tqdm(zip(all_files, all_labels), total=len(all_files))):
    feat = extract_temporal_features(fp)
    if feat is not None:
        sequences.append(feat)
        seq_labels.append(label)
    else:
        skipped += 1

sequences = np.array(sequences)
seq_labels = np.array(seq_labels)
print(f'\n✅ Extracted: {len(sequences)} sequences, skipped: {skipped}')
print(f'Shape: {sequences.shape}')  # (N, 10, 7)

# Save for reproducibility
np.save('/content/temporal_sequences.npy', sequences)
np.save('/content/temporal_labels.npy', seq_labels)

In [ ]:
# Step 7: Mamba SSM model (matches inference code architecture)

class SelectiveScanSSM(nn.Module):
    def __init__(self, d_model=64, state_dim=16, dt_rank=8, d_inner=32):
        super().__init__()
        self.d_model = d_model
        self.state_dim = state_dim
        self.dt_rank = dt_rank
        self.d_inner = d_inner

        self.in_proj = nn.Linear(d_model, d_inner * 2)
        self.A_log = nn.Parameter(torch.randn(d_inner, state_dim))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.x_proj = nn.Linear(d_inner, dt_rank + state_dim * 2, bias=False)
        self.dt_proj = nn.Linear(dt_rank, d_inner, bias=True)
        self.out_proj = nn.Linear(d_inner, d_model)
        self.norm = nn.LayerNorm(d_inner)
        self._init_weights()

    def _init_weights(self):
        A = torch.arange(1, self.state_dim + 1, dtype=torch.float32).unsqueeze(0).repeat(self.d_inner, 1)
        self.A_log.data = torch.log(A)
        nn.init.uniform_(self.dt_proj.bias, -4.0, -2.0)

    def forward(self, x):
        batch, seq_len, _ = x.shape
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)
        x_dbl = self.x_proj(x_in)
        dt, B, C = x_dbl.split([self.dt_rank, self.state_dim, self.state_dim], dim=-1)
        dt = torch.softplus(self.dt_proj(dt))
        A = -torch.exp(self.A_log)
        y = self._selective_scan(x_in, dt, A, B, C)
        y = self.norm(y)
        y = y * torch.silu(z)
        y = self.out_proj(y)
        return y

    def _selective_scan(self, u, dt, A, B, C):
        batch, seq_len, d_inner = u.shape
        h = torch.zeros(batch, d_inner, self.state_dim, device=u.device, dtype=u.dtype)
        outputs = []
        for t in range(seq_len):
            dt_t = dt[:, t, :].unsqueeze(-1)
            A_bar = torch.exp(A.unsqueeze(0) * dt_t)
            B_t = B[:, t, :].unsqueeze(1).expand(-1, d_inner, -1)
            B_bar = dt_t * B_t
            u_t = u[:, t, :].unsqueeze(-1)
            h = A_bar * h + B_bar * u_t
            C_t = C[:, t, :].unsqueeze(1).expand(-1, d_inner, -1)
            y_t = (C_t * h).sum(dim=-1) + self.D * u[:, t, :]
            outputs.append(y_t)
        return torch.stack(outputs, dim=1)


class MambaTemporalModel(nn.Module):
    def __init__(self, input_dim=7, d_model=64, n_layers=4, state_dim=16, d_inner=32, num_classes=8):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, 100, d_model) * 0.02)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'ssm': SelectiveScanSSM(d_model, state_dim, d_inner=d_inner),
                'norm': nn.LayerNorm(d_model),
                'ffn': nn.Sequential(
                    nn.Linear(d_model, d_model * 4),
                    nn.GELU(),
                    nn.Dropout(0.1),
                    nn.Linear(d_model * 4, d_model),
                    nn.Dropout(0.1),
                ),
                'ffn_norm': nn.LayerNorm(d_model),
            })
            for _ in range(n_layers)
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        batch, seq_len, _ = x.shape
        x = self.input_proj(x)
        x = x + self.pos_embed[:, :seq_len, :]
        for layer in self.layers:
            residual = x
            x = layer['norm'](x)
            x = layer['ssm'](x) + residual
            residual = x
            x = layer['ffn_norm'](x)
            x = layer['ffn'](x) + residual
        x = x.mean(dim=1)
        return self.head(x)

model = MambaTemporalModel(input_dim=FEATURE_DIM, num_classes=NUM_CLASSES).to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Mamba SSM parameters: {params:,}')

In [ ]:
# Step 8: Dataset and dataloaders

class TemporalDataset(Dataset):
    def __init__(self, sequences, labels, augment=False):
        self.sequences = torch.FloatTensor(sequences)
        self.labels = torch.LongTensor(labels)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = self.sequences[idx].clone()
        if self.augment:
            # Gaussian noise
            if np.random.random() < 0.4:
                seq += torch.randn_like(seq) * 0.05
            # Time masking (zero out a random window)
            if np.random.random() < 0.3:
                t = np.random.randint(0, seq.shape[0])
                seq[t] = 0
            # Feature scaling
            if np.random.random() < 0.3:
                scale = torch.FloatTensor(1).uniform_(0.9, 1.1)
                seq *= scale
        return seq, self.labels[idx]


train_seq, test_seq, train_lab, test_lab = train_test_split(
    sequences, seq_labels, test_size=0.15, random_state=42, stratify=seq_labels
)
train_seq, val_seq, train_lab, val_lab = train_test_split(
    train_seq, train_lab, test_size=0.15, random_state=42, stratify=train_lab
)

print(f'Train: {len(train_seq)}, Val: {len(val_seq)}, Test: {len(test_seq)}')

train_loader = DataLoader(TemporalDataset(train_seq, train_lab, augment=True), batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(TemporalDataset(val_seq, val_lab), batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(TemporalDataset(test_seq, test_lab), batch_size=64, shuffle=False, num_workers=2)

In [ ]:
# Step 9: Training loop

EPOCHS = 100
LR = 5e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_f1 = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(seqs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    train_loss /= len(train_loader)
    train_acc = 100.0 * correct / total

    # Validate
    model.eval()
    val_loss, all_preds, all_true = 0.0, [], []
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
            outputs = model(seqs)
            val_loss += criterion(outputs, labels).item()
            _, pred = outputs.max(1)
            all_preds.extend(pred.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
    val_loss /= len(val_loader)
    val_acc = 100.0 * np.mean(np.array(all_preds) == np.array(all_true))
    val_f1 = f1_score(all_true, all_preds, average='macro') * 100
    scheduler.step()

    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1})

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), '/content/temporal_mamba_best.pt')
        marker = ' ✅ BEST'
    else:
        marker = ''

    if epoch % 10 == 0 or marker:
        print(f'Epoch {epoch}/{EPOCHS} — Train: {train_loss:.4f}/{train_acc:.1f}% | '
              f'Val: {val_loss:.4f}/{val_acc:.1f}%/F1={val_f1:.1f}%{marker}')

print(f'\n🎉 Training complete! Best validation F1: {best_f1:.2f}%')

In [ ]:
# Step 10: Evaluate on test set

model.load_state_dict(torch.load('/content/temporal_mamba_best.pt', weights_only=True))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for seqs, labels in tqdm(test_loader, desc='Testing'):
        seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
        outputs = model(seqs)
        _, pred = outputs.max(1)
        all_preds.extend(pred.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

test_acc = 100.0 * np.mean(np.array(all_preds) == np.array(all_true))
test_f1 = f1_score(all_true, all_preds, average='macro') * 100
print(f'\n📊 Test Accuracy: {test_acc:.2f}%')
print(f'📊 Test F1 (macro): {test_f1:.2f}%')

print('\nClassification Report:')
report = classification_report(all_true, all_preds, target_names=EMOTION_LABELS, output_dict=True)
print(classification_report(all_true, all_preds, target_names=EMOTION_LABELS))

In [ ]:
# Step 11: Confusion matrix

cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Mamba SSM Temporal — Test Confusion Matrix (Acc={test_acc:.1f}%, F1={test_f1:.1f}%)')
plt.tight_layout()
plt.savefig('/content/temporal_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Step 12: Training curves

epochs = [h['epoch'] for h in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, [h['train_loss'] for h in history], label='Train Loss')
ax1.plot(epochs, [h['val_loss'] for h in history], label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, [h['train_acc'] for h in history], label='Train Acc')
ax2.plot(epochs, [h['val_acc'] for h in history], label='Val Acc')
ax2.plot(epochs, [h['val_f1'] for h in history], label='Val F1', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('%')
ax2.set_title('Accuracy & F1 Score')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/temporal_training_curves.png', dpi=150)
plt.show()

In [ ]:
# Step 13: Save results & download

results = {
    'model': 'MambaTemporalSSM',
    'datasets': ['RAVDESS', 'CREMA-D', 'TESS'],
    'total_samples': len(sequences),
    'seq_len': SEQ_LEN,
    'feature_dim': FEATURE_DIM,
    'num_classes': NUM_CLASSES,
    'classes': EMOTION_LABELS,
    'test_accuracy': round(test_acc, 2),
    'test_f1_macro': round(test_f1, 2),
    'best_val_f1': round(best_f1, 2),
    'epochs_trained': EPOCHS,
    'parameters': params,
    'classification_report': report,
}

with open('/content/temporal_results.json', 'w') as f:
    json.dump(results, f, indent=2)
with open('/content/temporal_training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('✅ Saved all results!')

from google.colab import files
files.download('/content/temporal_mamba_best.pt')
files.download('/content/temporal_results.json')
files.download('/content/temporal_confusion_matrix.png')
files.download('/content/temporal_training_curves.png')

print('\n📥 Downloaded! Place temporal_mamba_best.pt into:')
print('   mindtrack-ai/data/pretrained/temporal_mamba.pt')